# Model Evaluation

**Objective:** Comprehensive evaluation of trained deepfake detection models using standard metrics.

**Research Traceability:**
- Research Objective: Evaluate detection performance on standard metrics
- Methodology: Accuracy, Precision, Recall, F1, AUC-ROC, confusion matrix analysis
- Implementation: notebooks/06_evaluation.ipynb

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Test Results

In [ ]:
# Load experiment results
experiment_path = Path('../outputs/experiments')

def load_experiment(name: str) -> dict:
    """Load experiment results from JSON."""
    import json
    path = experiment_path / f'{name}.json'
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

# Load XceptionNet results
xception_results = load_experiment('xceptionnet_experiment')
efficientnet_results = load_experiment('efficientnet_experiment')

print("Available experiments:")
if xception_results:
    print(f"  XceptionNet: {xception_results.get('model_name', 'N/A')}")
if efficientnet_results:
    print(f"  EfficientNet: {efficientnet_results.get('model_name', 'N/A')}")

## 2. Confusion Matrix Analysis

In [ ]:
def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, 
                          title: str, save_path: str = None):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real', 'Fake'],
                yticklabels=['Real', 'Fake'])
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print classification report
    print(f"\n{title} Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))

## 3. ROC Curve Analysis

In [ ]:
def plot_roc_curves(models_data: dict, save_path: str = None):
    """Plot ROC curves for multiple models."""
    plt.figure(figsize=(8, 6))
    
    for name, data in models_data.items():
        y_true = data['y_true']
        y_scores = data['y_scores']
        
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend()
    plt.grid(True)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

## 4. Model Comparison

In [ ]:
def compare_models(results: list[dict]) -> None:
    """Compare multiple model results."""
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
    
    comparison = []
    for result in results:
        row = {'model': result.get('model_name', 'Unknown')}
        for metric in metrics:
            row[metric] = result.get('test_metrics', {}).get(metric, 0)
        comparison.append(row)
    
    import pandas as pd
    df = pd.DataFrame(comparison)
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 5))
    df.set_index('model')[metrics].plot(kind='bar', ax=ax)
    plt.title('Model Performance Comparison')
    plt.ylabel('Score')
    plt.ylim(0, 1.0)
    plt.legend(loc='lower right')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('../outputs/plots/model_comparison.png', dpi=150)
    plt.show()
    
    print("\nModel Comparison Table:")
    print(df.to_string(index=False))

## 5. Per-Class Analysis

In [ ]:
def analyze_per_class_performance(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Analyze per-class performance metrics."""
    cm = confusion_matrix(y_true, y_pred)
    
    tn, fp, fn, tp = cm.ravel()
    
    real_precision = tn / (tn + fn) if (tn + fn) > 0 else 0
    real_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    fake_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    fake_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    return {
        'real': {'precision': real_precision, 'recall': real_recall},
        'fake': {'precision': fake_precision, 'recall': fake_recall},
        'true_negatives': tn,
        'false_positives': fp,
        'false_negatives': fn,
        'true_positives': tp,
    }

## Summary
- Confusion matrix visualization
- ROC curve comparison across models
- Per-class performance analysis
- Model comparison summary table

## Next Steps
- Generate thesis assets: `07_inference_demo.ipynb`
- Export results to CSV/JSON for Chapter 4